# 06: Qwen 3B QLoRA 모델 평가, Colab

Drive 연결 후 같은 실험의 데이터, 분할 기록, 학습 실행 기록, 어댑터와 토크나이저를 사용합니다. Drive 연결은 학습 노트북의 연결 셀을 사용합니다. 기본 모델은 FP16으로 불러와 어댑터를 병합합니다.


## 1. 파라미터와 환경


In [ ]:
RUNTIME_PARAMS = {
    'prefix_path': '/drive/MyDrive/SiteBoard/06/',
    'model_name': 'Qwen/Qwen2.5-3B-Instruct',
    'model_revision': 'aa8e72537993ba99e69dfaafa59ed015b17504d1',
    'adapter_path': 'models',
    'test_file': 'test.jsonl',
    'split_meta_file': 'split_meta.json',
    'run_config_file': 'result/run_config.json',
    'report_file': 'result/evaluation.json',
    'device': 'cuda',
}

HYPER_PARAMS = {
    # 기본 모델은 FP16으로 불러온 뒤 LoRA 어댑터를 병합합니다.
    'model_loading': {
        'attn_implementation': 'sdpa',
        'low_cpu_mem_usage': True,
    },
    'tokenizer': {
        'padding_side': 'left',  # 길이가 다른 프롬프트를 배치로 생성합니다.
    },
    'evaluation': {
        'batch_size': 2,
        'max_new_tokens': 512,
        'max_input_tokens': 1024,
        'do_sample': False,
    },
}


In [ ]:
# Colab GPU 런타임에서 실행하세요. 설치 후 필요하면 런타임을 재시작하세요.
!pip install "torch>=2.2.0" "transformers>=4.46.0,<5" "peft>=0.13.0,<0.18" "accelerate>=0.29.0,<2" "bitsandbytes>=0.43.0,<1" "tqdm>=4.66.0"


## 2. 라이브러리


In [ ]:
import hashlib
import json
import math
import platform
import re
import sys
from collections import Counter
from copy import deepcopy
from pathlib import Path

import torch
from peft import PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


## 3. 결과 저장과 파일 해시


In [ ]:
def runtime_path(field):
    return str(Path(RUNTIME_PARAMS["prefix_path"]) / RUNTIME_PARAMS[field])


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("x", encoding="utf-8") as stream:
        json.dump(value, stream, ensure_ascii=False, indent=2)
        stream.write("\n")


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


## 4. 평가 데이터 읽기


In [ ]:
def read_jsonl(path):
    path = Path(path)
    records = []
    with path.open("r", encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSON 문법 오류: {path}:{line_number}") from exc
            messages = record.get("messages") if isinstance(record, dict) else None
            if not isinstance(messages, list) or len(messages) != 3:
                raise ValueError(f"system, user, assistant 메시지 3개가 필요합니다: {path}:{line_number}")
            for message, role in zip(messages, ("system", "user", "assistant")):
                if (not isinstance(message, dict) or message.get("role") != role
                        or not isinstance(message.get("content"), str)):
                    raise ValueError(f"메시지 역할과 문자열 content를 확인하세요: {path}:{line_number}")
            records.append(record)
    if not records:
        raise ValueError(f"데이터가 비어 있습니다: {path}")
    return records



## 5. 엄격한 채점과 값 토큰 신뢰도

작업 순서는 무시하고 중복 개수는 유지합니다. 이전의 느슨한 JSON 채점과 기준이 다릅니다. 값에 대응하는 토큰이 없거나 확인할 수 없으면 신뢰도는 `null`로 저장합니다.


In [ ]:
def unique_object(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"중복 JSON 키: {key}")
        result[key] = value
    return result


def parse_prediction(text):
    """필드와 값 타입을 검사하고, 유효하지 않은 출력은 None으로 반환합니다."""
    try:
        data = json.loads(text, object_pairs_hook=unique_object)
        if not isinstance(data, list):
            return None
        for item in data:
            if not isinstance(item, dict) or set(item) != {"workType", "spec", "quantity"}:
                return None
            if not isinstance(item["workType"], str) or not item["workType"].strip():
                return None
            if not isinstance(item["spec"], str):
                return None
            item["workType"].encode("utf-8")
            item["spec"].encode("utf-8")
            if type(item["quantity"]) not in (int, float) or not math.isfinite(item["quantity"]):
                return None
        return data
    except (ValueError, TypeError, OverflowError, RecursionError):
        return None


def parse_and_normalize(text):
    data = parse_prediction(text)
    if data is None:
        return None
    return [(item["workType"].strip(), item["spec"].strip(), round(item["quantity"], 4)) for item in data]


def build_token_spans(tokenizer, token_ids, probabilities, pred_text):
    """Qwen 바이트 토큰을 복원해 한글 분할 토큰에도 같은 값 구간을 연결합니다."""
    visible = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
    missing = [value for value in range(256) if value not in visible]
    byte_decoder = dict(zip(map(chr, visible + list(range(256, 256 + len(missing)))), visible + missing))
    spans = []
    restored = bytearray()
    try:
        for token, probability in zip(tokenizer.convert_ids_to_tokens(token_ids), probabilities):
            start = len(restored)
            restored.extend(byte_decoder[char] for char in token)
            spans.append((start, len(restored), probability))
    except KeyError:
        return None
    if len(spans) != len(token_ids) or bytes(restored) != pred_text.encode("utf-8"):
        return None
    return spans


def extract_values_with_confidence(pred_text, token_spans):
    """값과 겹치는 생성 토큰을 집계하며 대응이 미확인인 값은 None으로 둡니다."""
    data = parse_prediction(pred_text)
    if data is None:
        return [], None, None
    predictions = [dict(item) for item in data]
    for item in predictions:
        for field in ("workType", "spec", "quantity"):
            item[f"{field}_confidence"] = None
    if token_spans is None:
        return predictions, None, None

    # JSON 문자열 전체를 먼저 매칭해 값 내부의 키처럼 보이는 문자열을 제외합니다.
    string = r'"(?:[^"\\\x00-\x1f]|\\(?:["\\/bfnrt]|u[0-9a-fA-F]{4}))*"'
    number = r'-?(?:0|[1-9][0-9]*)(?:\.[0-9]+)?(?:[eE][+-]?[0-9]+)?'
    fields = list(re.finditer(rf'({string})\s*:\s*({string}|{number})|{string}', pred_text))
    fields = [match for match in fields if match.group(1) is not None]
    if len(fields) != 3 * len(data):
        return predictions, None, None
    byte_offsets = [0]
    for char in pred_text:
        byte_offsets.append(byte_offsets[-1] + len(char.encode("utf-8")))
    value_indices = set()
    for index, match in enumerate(fields):
        field = json.loads(match.group(1))
        start, end = match.span(2)
        if pred_text[start] == '"':
            start, end = start + 1, end - 1
        selected = {i for i, (left, right, _) in enumerate(token_spans)
                    if max(byte_offsets[start], left) < min(byte_offsets[end], right)}
        if selected:
            probabilities = [token_spans[i][2] for i in selected]
            predictions[index // 3][f"{field}_confidence"] = sum(probabilities) / len(probabilities)
            value_indices.update(selected)
    probabilities = [token_spans[i][2] for i in value_indices]
    return (predictions, sum(probabilities) / len(probabilities), min(probabilities)) if probabilities else (predictions, None, None)


def summarize_cases(cases):
    values = [case["value_min_confidence"] for case in cases if case["value_min_confidence"] is not None]
    averages = [case["value_avg_confidence"] for case in cases if case["value_avg_confidence"] is not None]
    correct = sum(case["is_exact_match"] for case in cases)
    return {
        "total": len(cases),
        "correct": correct,
        "accuracy": 100.0 * correct / len(cases) if cases else None,
        "value_min_confidence": min(values) if values else None,
        "value_avg_confidence": sum(averages) / len(averages) if averages else None,
        "confidence_count": len(values),
        "confidence_missing_count": len(cases) - len(values),
    }


## 6. 모델 로딩과 입력 확인


In [ ]:
def load_tokenizer():
    if RUNTIME_PARAMS["device"] != "cuda" or not torch.cuda.is_available():
        raise RuntimeError("CUDA 사용이 가능한 GPU와 PyTorch 설치 및 노트북 커널을 확인하세요.")
    adapter = Path(runtime_path("adapter_path"))
    for name in ("adapter_config.json", "adapter_model.safetensors", "tokenizer.json", "tokenizer_config.json"):
        if not (adapter / name).is_file():
            raise FileNotFoundError(f"필수 모델 파일이 없습니다: {adapter / name}")
    tokenizer = AutoTokenizer.from_pretrained(adapter, **HYPER_PARAMS["tokenizer"])
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_model():
    model = AutoModelForCausalLM.from_pretrained(
        RUNTIME_PARAMS["model_name"],
        revision=RUNTIME_PARAMS["model_revision"],
        torch_dtype=torch.float16,
        device_map={"": RUNTIME_PARAMS["device"]},
        **HYPER_PARAMS["model_loading"],
    )
    model = PeftModel.from_pretrained(model, runtime_path("adapter_path")).merge_and_unload()
    model.config.use_cache = True
    model.eval()
    return model


In [ ]:
def prepare_evaluation(tokenizer):
    samples = read_jsonl(runtime_path("test_file"))
    with Path(runtime_path("run_config_file")).open(encoding="utf-8") as stream:
        run_config = json.load(stream)
    execution = run_config["execution"]
    if (run_config["status"] != "training_completed"
            or execution["base_model"] != RUNTIME_PARAMS["model_name"]
            or execution["base_model_revision"] != RUNTIME_PARAMS["model_revision"]
            or run_config["prepared_data"]["test_count"] != len(samples)):
        raise ValueError("모델 이름, revision, 평가 건수 또는 학습 완료 기록이 다릅니다.")
    test_hash = file_sha256(runtime_path("test_file"))
    expected = [value for key, value in run_config["prepared_file_hashes"].items()
                if Path(key.replace("\\", "/")).name == Path(RUNTIME_PARAMS["test_file"]).name]
    if len(expected) != 1 or expected[0] != test_hash:
        raise ValueError("평가 파일이 학습 당시 test 파일과 다릅니다.")
    with Path(runtime_path("split_meta_file")).open(encoding="utf-8") as stream:
        split = json.load(stream)
    subsets = {name: set(subset["indices"]) for name, subset in split["test"]["subsets"].items()}
    identifiers, source_indices, prompts = set(), set(), []
    for index, sample in enumerate(samples, 1):
        metadata = sample.get("metadata", {})
        if not all(key in metadata for key in ("site", "image_id", "source_index", "test_type")):
            raise ValueError(f"평가 식별자 또는 유형이 없습니다: {index}")
        identifier = (metadata["site"], metadata["image_id"])
        source_index = metadata["source_index"]
        test_type = metadata["test_type"]
        if (identifier in identifiers or source_index in source_indices
                or test_type not in subsets or source_index not in subsets[test_type]):
            raise ValueError(f"평가 식별자 중복 또는 분할 유형 불일치: {index}")
        identifiers.add(identifier)
        source_indices.add(source_index)
        if parse_prediction(sample["messages"][2]["content"]) is None:
            raise ValueError(f"정답 JSON 스키마 오류: {index}")
        prompt = tokenizer.apply_chat_template(sample["messages"][:2], tokenize=False, add_generation_prompt=True)
        length = len(tokenizer.encode(prompt, add_special_tokens=False))
        if length > HYPER_PARAMS["evaluation"]["max_input_tokens"]:
            raise ValueError(f"입력 토큰 상한 초과: {index}, {length}토큰")
        prompts.append(prompt)
    if (source_indices != set(split["test"]["indices"])
            or any({s["metadata"]["source_index"] for s in samples if s["metadata"]["test_type"] == name} != indices
                   for name, indices in subsets.items())):
        raise ValueError("평가 인덱스가 분할 기록과 다릅니다.")
    paths = [runtime_path(key) for key in ("test_file", "split_meta_file", "run_config_file")]
    adapter = Path(runtime_path("adapter_path"))
    paths.extend(str(adapter / name) for name in ("adapter_config.json", "adapter_model.safetensors", "tokenizer.json", "tokenizer_config.json"))
    return samples, prompts, {path: file_sha256(path) for path in paths}


## 7. 실행 정보와 평가


In [ ]:
def collect_execution_metadata(model=None):
    from importlib.metadata import PackageNotFoundError, version

    packages = {}
    for name in ("torch", "transformers", "peft", "accelerate", "bitsandbytes"):
        try:
            packages[name] = version(name)
        except PackageNotFoundError:
            packages[name] = None
    return {
        "python": platform.python_version(),
        "python_executable": sys.executable,
        "platform": platform.platform(),
        "packages": packages,
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "cuda": torch.version.cuda,
        "base_model": RUNTIME_PARAMS["model_name"],
        "base_model_revision": getattr(getattr(model, "config", None), "_commit_hash", None)
            or RUNTIME_PARAMS.get("model_revision"),
    }



In [ ]:
def evaluate():
    report_path = Path(runtime_path("report_file"))
    if report_path.exists():
        raise FileExistsError(f"기존 평가 결과가 있습니다: {report_path}")
    settings = HYPER_PARAMS["evaluation"]
    if any(type(settings[key]) is not int or settings[key] <= 0
           for key in ("batch_size", "max_input_tokens", "max_new_tokens")):
        raise ValueError("평가 배치와 토큰 상한은 양의 정수여야 합니다.")
    tokenizer = load_tokenizer()
    samples, prompts, file_hashes = prepare_evaluation(tokenizer)
    model = load_model()
    eos_ids = {tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")}
    eos_ids.discard(None)
    case_results = []
    print(f"평가 {len(samples)}건 / GPU: {torch.cuda.get_device_name(0)}")

    for start in tqdm(range(0, len(samples), settings["batch_size"]), desc="Evaluation"):
        batch = samples[start:start + settings["batch_size"]]
        inputs = tokenizer(prompts[start:start + len(batch)], return_tensors="pt", padding=True,
                           truncation=False, add_special_tokens=False).to(model.device)
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=settings["max_new_tokens"],
                do_sample=settings["do_sample"],
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=sorted(eos_ids),
                return_dict_in_generate=True,
                output_scores=True,
            )
        generated = outputs.sequences[:, inputs["input_ids"].shape[1]:]
        for offset, sample in enumerate(batch):
            token_ids = generated[offset].tolist()
            end = next((i for i, token_id in enumerate(token_ids)
                        if token_id in eos_ids or token_id == tokenizer.pad_token_id), len(token_ids))
            terminated = end < len(token_ids) and token_ids[end] in eos_ids
            token_ids = token_ids[:end]
            pred_text = tokenizer.decode(token_ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)
            probabilities = [torch.softmax(outputs.scores[i][offset].float(), dim=-1)[token_id].item()
                             for i, token_id in enumerate(token_ids)]
            token_spans = build_token_spans(tokenizer, token_ids, probabilities, pred_text)
            prediction, avg_confidence, min_confidence = extract_values_with_confidence(pred_text, token_spans)
            ground_truth = sample["messages"][2]["content"]
            norm_pred = parse_and_normalize(pred_text)
            is_match = norm_pred is not None and Counter(norm_pred) == Counter(parse_and_normalize(ground_truth))
            metadata = sample["metadata"]
            case_results.append({
                "case_id": start + offset + 1,
                **{key: metadata[key] for key in ("site", "image_id", "source_index", "test_type")},
                "input_user": sample["messages"][1]["content"],
                "ground_truth_raw": ground_truth,
                "prediction_raw": pred_text,
                "prediction": prediction,
                "is_valid_json_schema": norm_pred is not None,
                "is_exact_match": is_match,
                "output_limit_reached": not terminated and end >= settings["max_new_tokens"],
                "value_avg_confidence": avg_confidence,
                "value_min_confidence": min_confidence,
            })
        # 다음 배치를 생성하기 전에 어휘 점수를 해제합니다.
        del outputs, inputs, generated

    overall = summarize_cases(case_results)
    by_type = {name: summarize_cases([case for case in case_results if case["test_type"] == name])
               for name in sorted({case["test_type"] for case in case_results})}
    if (sum(group["total"] for group in by_type.values()) != overall["total"]
            or sum(group["correct"] for group in by_type.values()) != overall["correct"]):
        raise ValueError("평가 유형별 합계가 전체와 일치하지 않습니다.")
    report = {
        "model": RUNTIME_PARAMS["model_name"],
        "adapter": runtime_path("adapter_path"),
        "device": torch.cuda.get_device_name(0),
        "dtype": str(model.dtype),
        "test_file": runtime_path("test_file"),
        "runtime_params": deepcopy(RUNTIME_PARAMS),
        "hyper_params": deepcopy(HYPER_PARAMS),
        "batch_size": settings["batch_size"],
        "scoring_version": "strict_schema_multiset_v1",
        "confidence_definition": "Generated tokens overlapping workType/spec/quantity value bytes; unique tokens per case. Group min includes correct and incorrect cases with known confidence; null cases are excluded and counted. Avg is the mean of known case averages.",
        "file_hashes": file_hashes,
        "execution": collect_execution_metadata(model),
        **overall,
        "correct_value_avg_confidence": summarize_cases([case for case in case_results if case["is_exact_match"]])["value_avg_confidence"],
        "incorrect_value_avg_confidence": summarize_cases([case for case in case_results if not case["is_exact_match"]])["value_avg_confidence"],
        "by_test_type": by_type,
        "cases": case_results,
    }
    # 평가 도중 결과 파일이 만들어진 경우에도 덮어쓰지 않습니다.
    if report_path.exists():
        raise FileExistsError(f"기존 평가 결과가 있습니다: {report_path}")
    write_json(report_path, report)
    print(f"완전 일치율: {overall['accuracy']:.2f}% ({overall['correct']}/{overall['total']})")
    print(f"평가 결과 저장: {report_path}")
    return report


## 8. 설정 확인 후 평가

학습 커널을 종료해 GPU 메모리를 확보한 뒤 실행합니다. 입력 토큰 상한을 넘으면 중단하고, 출력 상한 도달 여부는 케이스에 기록합니다. FP16 병합 평가는 4비트 학습과 수치 차이가 있을 수 있습니다. 기존 결과는 덮어쓰지 않습니다.


In [ ]:
print("RUNTIME_PARAMS:", json.dumps(RUNTIME_PARAMS, ensure_ascii=False, indent=2))
print("HYPER_PARAMS:", json.dumps(HYPER_PARAMS, ensure_ascii=False, indent=2))


In [ ]:
report = evaluate()
